## Patient Journey - Diagnosis
#### Data source used
* PHCDW.PHCDW_DSAA.IQV_LAAD_DX_VW
* PHCDW.PHCDW_STG_CAR.STG_IQVIA_LAAD_DIM_DIAGNOSIS
* PHCDW.PHCDW_STG_CAR.STG_IQVIA_LAAD_DIM_PRODUCT 

#### Steps:
1. Query CKD diagnosis claims data after 2016-01-01
2. Get date level diagnosis stage records (get the daily maximium if multiple records for one date, forward fill for missing between two diagnosis) for each patient
3. Save table to databricks

#### Notes:
1. Time frame: SVC_DT >= '2017-01-01' 
2. CKD stage:     
    * when CATEGORY = 'CHRONIC KIDNEY DISEASE, STAGE 1'  then 1
    * when CATEGORY = 'CHRONIC KIDNEY DISEASE, STAGE 2 (MILD)'  then 2
    * when CATEGORY = 'CHRONIC KIDNEY DISEASE, STAGE 3 (MODERATE)'  then 3
    * when CATEGORY = 'CHRONIC KIDNEY DISEASE, STAGE 4 (SEVERE)'  then 4
    * when CATEGORY like '%STAGE 5%'  then 5
3. Check the the end_date of the data, now we set end_date_limit = Max(SVC_DT)

## 1. Setup Snowflake connection and load data

In [0]:
%run "/Workspace/Users/wenjie.chen@bayer.com/kerendia_nba_3_0_develop/NBA 3.0 Ops - Pipeline/00_Helper_Notebooks/00_connection"

### Load Libraries

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


### Get Data from Scope

[REDACTED]

[REDACTED]

[REDACTED]

[REDACTED]



### Define Snowflake Connection Function
Choose appropiate below items based on the Brand
- Role
- Warehouse
- Schema
- Database

In [0]:
%run "./00_cohort_config"

# Cohort Configuration

Shared config notebook for the 6 patient journey validation notebooks. 
Defines the offset, data boundaries, cohort labels (C1–C22), and the cohort assignment function.

**Usage:** Each patient journey notebook should `%run` this notebook early on.

OFFSET_DAY      = 0
DATA_START_DATE = 2023-01-01
COHORT_ORIGIN   = 2025-01-01
COHORT_LAST     = 2026-10-01


cohort_label,cohort_ref_month,cohort_start,cohort_end
C1,2025-01-01,2025-01-01,2025-02-01
C2,2025-02-01,2025-02-01,2025-03-01
C3,2025-03-01,2025-03-01,2025-04-01
C4,2025-04-01,2025-04-01,2025-05-01
C5,2025-05-01,2025-05-01,2025-06-01
C6,2025-06-01,2025-06-01,2025-07-01
C7,2025-07-01,2025-07-01,2025-08-01
C8,2025-08-01,2025-08-01,2025-09-01
C9,2025-09-01,2025-09-01,2025-10-01
C10,2025-10-01,2025-10-01,2025-11-01



22 cohorts defined (C1 – C22)


Testing with current OFFSET_DAY = 0
-------------------------------------------------------
  2025-01-01 → 2025-01-01  (expected 2025-01-01)  [PASS]

Config notebook ready. All 6 patient journey notebooks can %run this.


In [0]:
import pandas as pd
import pyspark.sql.functions as F
df_dx = spark.read.format("delta").load("/Volumes/ph-com-ai-us-prod-catalog-2553213839599380/kerendia_nba_3/kerendia_nba_3_volume/hcp_level_model/ops_pipeline/query_output/pat_journey_dx")

# Filter to data >= DATA_START_DATE (from config)
df_dx = df_dx.filter(F.col("SVC_DT") >= DATA_START_DATE)

# Validation filter: 10 test HCPs (from config)
df_dx, test_patients = apply_hcp_filter(df_dx)

pd.DataFrame(df_dx.columns).display()

Filtered to 821 patients across 10 HCPs


0
PROVIDER_ID
NPI_NUMBER
PATIENT_ID
SVC_DT
DX_STAGE_TEXT
DX_STAGE
PULL_DT


QC

In [0]:
# df_dx_dev = spark.read.format("parquet").load("/Volumes/ph-com-ai-us-prod-catalog-2553213839599380/kerendia_nba_3/kerendia_nba_3_volume/hcp_level_model/dev_folder/query_output/pat_journey_dx.parquet")
# pd.DataFrame(df_dx_dev.columns).display()

0
NPI_NUMBER
PATIENT_ID
SVC_DT
DX_STAGE_TEXT
DX_STAGE
PROVIDER_ID
PULL_DT


In [0]:
# overlap_patients = df_dx.select("PATIENT_ID").distinct().join(df_dx_dev.select("PATIENT_ID").distinct(), on = "PATIENT_ID", how = "inner")
# overlap_patients.show(5)

+----------+
|PATIENT_ID|
+----------+
| 742775178|
| 154042682|
| 761719184|
|3312682746|
| 792561155|
+----------+
only showing top 5 rows


In [0]:
# df1 = df_dx.select("PATIENT_ID", "SVC_DT", "DX_STAGE")
# df1 = df1.withColumnRenamed("DX_STAGE", "DX_STAGE_OPS")
# df2 = df_dx_dev.select("PATIENT_ID", "SVC_DT", "DX_STAGE")
# df2 = df2.withColumnRenamed("DX_STAGE", "DX_STAGE_DEV")
# df_comp = df1.join(df2, on = ['PATIENT_ID', 'SVC_DT'], how = "left")
# # print(df_comp.filter(F.col('DX_STAGE_OPS') != F.col('DX_STAGE_DEV')).select('PATIENT_ID').distinct().count())
# df_comp.filter(F.col('DX_STAGE_OPS') != F.col('DX_STAGE_DEV')).display()

PATIENT_ID,SVC_DT,DX_STAGE_OPS,DX_STAGE_DEV
1712184124,2024-01-10,0,4
1007463648,2021-02-01,4,0
1712184124,2024-01-10,0,4
360540092,2022-04-20,4,0
687470160,2022-01-31,0,4
302301868,2023-07-11,0,4
138451047,2021-03-09,4,0
1730726946,2021-02-05,4,0
874780639,2021-01-26,2,0
1005663137,2024-03-05,0,4


## 2. Data cleanup and processing

In [0]:
from datetime import datetime
from pyspark.sql.types import DateType, StructType, StructField, StringType
from pyspark.sql import functions as F, Window
import sys

# Function to generate cohort-month reference dates (1st of each month)
def generate_month_range(start, end):
    current = datetime.strptime(start, "%Y-%m-%d")
    end = datetime.strptime(end, "%Y-%m-%d")
    months = []
    while current <= end:
        months.append(current.strftime("%Y-%m-01"))
        if current.month == 12:
            current = current.replace(year=current.year + 1, month=1, day=1)
        else:
            current = current.replace(month=current.month + 1, day=1)
    return months

# Step 1: Get min and max SVC_DT, convert to cohort-months using offset
min_dt, max_dt = df_dx.agg(F.min("SVC_DT"), F.max("SVC_DT")).first()
if OFFSET_DAY == 0:
    min_month_str = min_dt.strftime("%Y-%m-01")
    max_month_str = max_dt.strftime("%Y-%m-01")
else:
    from datetime import timedelta
    min_shifted = min_dt - timedelta(days=OFFSET_DAY - 1)
    max_shifted = max_dt - timedelta(days=OFFSET_DAY - 1)
    min_month_str = min_shifted.strftime("%Y-%m-01")
    max_month_str = max_shifted.strftime("%Y-%m-01")

# Step 2: Create DataFrame of all possible cohort-months for all PATIENT_IDs
all_months = generate_month_range(min_month_str, max_month_str)
months_schema = StructType([StructField("COHORT_MONTH", StringType(), True)])
months_df = spark.createDataFrame([(m,) for m in all_months], schema=months_schema)

patient_ids = df_dx.select("PATIENT_ID").distinct()
patient_months_df = patient_ids.crossJoin(months_df)

# Step 3: Map SVC_DT to cohort-month and group
df_dx_monthly = (
    df_dx
    .withColumn("COHORT_MONTH", F.date_format(assign_cohort_month(F.col("SVC_DT")), "yyyy-MM-01"))
    .groupBy("PATIENT_ID", "COHORT_MONTH")
    .agg(F.collect_set("DX_STAGE").alias("DX_STAGE"))
)

# Step 4: Join complete patient-cohort-month grid with aggregated data
df_dx_monthly = (
    patient_months_df
    .join(df_dx_monthly, on=["PATIENT_ID", "COHORT_MONTH"], how="left")
)

# Step 5: Compute max stage per cohort-month
df_dx_monthly = (
    df_dx_monthly
    .withColumn("DX_STAGE_MAX", F.array_max(F.col("DX_STAGE")))
    .withColumn("DX_STAGE_MAX", F.when(F.col("DX_STAGE_MAX") == 0, None).otherwise(F.col("DX_STAGE_MAX")))
)

# Step 6: Define forward fill window
w_forward = (
    Window.partitionBy("PATIENT_ID")
    .orderBy("COHORT_MONTH")
    .rowsBetween(-sys.maxsize, 0)
)

# Step 7: Forward fill DX_STAGE_MAX
df_dx_monthly_filled = (
    df_dx_monthly
    .withColumn("DX_STAGE_UPDATED", F.last("DX_STAGE_MAX", ignorenulls=True).over(w_forward))
)

# Combine and fill nulls with 0
df_dx_combine = (
    df_dx_monthly
    .join(df_dx_monthly_filled.select("PATIENT_ID", "COHORT_MONTH", "DX_STAGE_UPDATED"), 
          on=["PATIENT_ID", "COHORT_MONTH"], how="left")
    .na.fill(value=0, subset=["DX_STAGE_UPDATED"])
)

# Join cohort labels (C1-C22); rows before C1 get null label (lookback)
df_dx_combine = df_dx_combine.join(
    df_cohort_ref.select(
        F.date_format("cohort_ref_month", "yyyy-MM-01").alias("_ref_month_str"),
        "cohort_label"
    ),
    df_dx_combine.COHORT_MONTH == F.col("_ref_month_str"),
    "left"
).drop("_ref_month_str")

df_dx_combine.display()

PATIENT_ID,COHORT_MONTH,DX_STAGE,DX_STAGE_MAX,DX_STAGE_UPDATED,cohort_label
1984102749,2023-01-01,null,null,0,null
689020749,2025-04-01,null,null,0,C4
350305614,2025-01-01,null,null,4,C1
164583916,2026-07-01,null,null,4,C19
1071208306,2023-06-01,null,null,0,null
1105939024,2024-02-01,null,null,0,null
7925779953,2025-01-01,null,null,1,C1
1083457388,2023-12-01,null,null,0,null
340430802,2024-11-01,null,null,4,null
1887312251,2026-02-01,null,null,0,C14


In [0]:
df_dx_combine = df_dx_combine.withColumnRenamed("COHORT_MONTH", "DATE")


df_dx_combine = df_dx_combine.drop("DX_STAGE_MAX")


In [0]:
print('Shape:',df_dx_combine.count(), len(df_dx_combine.columns))
print('# Unique HCPs:',df_dx_combine.select("PATIENT_ID").distinct().count())
print('Min Date:',df_dx_combine.agg({"DATE": "min"}).collect()[0][0])
print('Max Date:', df_dx_combine.agg({"DATE": "max"}).collect()[0][0])

Shape: 36124 5
# Unique HCPs: 821
Min Date: 2023-01-01
Max Date: 2026-08-01


In [0]:
df_dx_combine.printSchema()

root
 |-- PATIENT_ID: string (nullable = true)
 |-- DATE: string (nullable = true)
 |-- DX_STAGE: array (nullable = true)
 |    |-- element: decimal(1,0) (containsNull = false)
 |-- DX_STAGE_UPDATED: decimal(1,0) (nullable = true)
 |-- cohort_label: string (nullable = true)



In [0]:
df_dx_combine = df_dx_combine.withColumn("DATE", F.to_date(F.col("DATE"), "yyyy-MM-dd"))

In [0]:
# COMPARISON: offset=0 validation vs production
base = "/Volumes/ph-com-ai-us-prod-catalog-2553213839599380/kerendia_nba_3/kerendia_nba_3_volume/hcp_level_model/ops_pipeline/query_output"
prod_dx = spark.read.format("delta").load(f"{base}/df_dx_combine")
val_pats = [r.PATIENT_ID for r in df_dx_combine.select("PATIENT_ID").distinct().collect()]
prod_dx_f = prod_dx.filter(F.col("PATIENT_ID").isin(val_pats))

val_c = df_dx_combine.count()
prod_c = prod_dx_f.count()
print(f"Validation (offset=0): {val_c:,}")
print(f"Production (same pts): {prod_c:,}")
print(f"Difference:            {val_c - prod_c:+,}")

val_k = df_dx_combine.select("PATIENT_ID", "DATE")
prod_k = prod_dx_f.select("PATIENT_ID", F.col("DATE").cast("date").alias("DATE"))
only_val = val_k.subtract(prod_k).count()
only_prod = prod_k.subtract(val_k).count()
print(f"\nRows only in validation: {only_val}")
print(f"Rows only in production: {only_prod}")
prod_only_before = prod_k.subtract(val_k).filter(F.col("DATE") < DATA_START_DATE).count()
print(f"  Prod-only before {DATA_START_DATE}: {prod_only_before}")
print(f"  Prod-only on/after {DATA_START_DATE}: {only_prod - prod_only_before}")

# Stage diffs on overlapping rows
common = df_dx_combine.alias("v").join(
    prod_dx_f.withColumn("DATE", F.col("DATE").cast("date")).alias("p"),
    (F.col("v.PATIENT_ID") == F.col("p.PATIENT_ID")) & (F.col("v.DATE") == F.col("p.DATE")), "inner"
)
diffs = common.filter(
    (F.col("v.DX_STAGE_UPDATED") != F.col("p.DX_STAGE_UPDATED")) |
    (F.col("v.DX_STAGE_UPDATED").isNull() != F.col("p.DX_STAGE_UPDATED").isNull())
).count()
print(f"\nOverlapping rows: {common.count():,}")
print(f"Stage diffs: {diffs}")

Validation (offset=0): 36,124
Production (same pts): 55,828
Difference:            -19,704

Rows only in validation: 0
Rows only in production: 19704
  Prod-only before 2023-01-01: 19704
  Prod-only on/after 2023-01-01: 0

Overlapping rows: 36,124
Stage diffs: 2026


In [0]:
# VALIDATION: write to a separate path so we don't overwrite production data
df_dx_combine.write.mode("overwrite").format("delta").save("/Volumes/ph-com-ai-us-prod-catalog-2553213839599380/kerendia_nba_3/kerendia_nba_3_volume/hcp_level_model/ops_pipeline/query_output/kerendia_nba_3_1_dx_journey_validation")